# Limpieza y preprocesamiento de datos: DB AFAC (Aerolíneas)

Este notebook documenta el proceso de limpieza y preparación del dataset
**DB_AFAC**, proveniente de la Agencia Federal de Aviación Civil (AFAC).
El dataset contiene registros de pasajeros transportados por aerolínea,
aeropuerto, tipo de vuelo (nacional/internacional) y periodo.

A diferencia de los datasets de Residencia y Nacionalidad, esta fuente
no proviene de encuestas a viajeros, sino de registros operativos de las
aerolíneas, lo que permite cruzar el flujo turístico con la oferta de asientos
y rutas del mercado aéreo mexicano.

El producto final de este notebook es **`df_afac_modelo`**: dataset filtrado
a vuelos internacionales, sin años atípicos, limpio y normalizado,
listo para el pipeline de modelado.

In [ ]:
!pip install openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("Entorno configurado correctamente para DB_AFAC.")

Entorno configurado correctamente para DB_AFAC.


## 1. Carga del dataset

El dataset se descarga desde el repositorio del proyecto en GitHub
en formato `.xlsx`. Se imprime la dimensión resultante para verificar
que la lectura fue exitosa.

In [ ]:
url_afac = "https://github.com/Montiel-Oscar/ciencia_de_datos_2026_2/raw/283d4ed6068f70af81e280f88ceb73cc09c7432a/proyecto_borrador/datasets/DB_AFAC.xlsx"

print("Descargando el dataset de Aerolíneas (AFAC)...")
df_afac = pd.read_excel(url_afac, engine="openpyxl")

print("¡Carga exitosa!")
print(f"Forma inicial: {df_afac.shape[0]:,} filas x {df_afac.shape[1]} columnas")
display(df_afac.head(3))

Descargando el dataset de Aerolíneas (AFAC)...
¡Carga exitosa!
Forma inicial: 18,804 filas x 12 columnas


,Año,Tipo,Servicio,Reg,Aerolinea,Id_mes,Mes,Pasajeros,fecha,Tipo.1,Servicio.1,Aerolínea
0,2016,INT,FLET,CAN,Air Transat,1,Ene,52930,42370,Internacional,Fletamiento,Canadiense
1,2016,INT,FLET,CAN,Cargojet Airways Ltd,1,Ene,0,42370,Internacional,Fletamiento,Canadiense
2,2016,INT,FLET,CAN,Sunwing Airlines,1,Ene,33932,42370,Internacional,Fletamiento,Canadiense


## 2. Exploración general

Se genera el resumen técnico del dataset: tipo de dato por columna,
valores nulos y cardinalidad. Adicionalmente se listan los nombres
exactos de las columnas para identificar redundancias y columnas duplicadas
(por ejemplo, `Tipo` y `Tipo.1`, o `Aerolínea` y `Aerolinea`),
que serán gestionadas en la siguiente etapa.

In [ ]:
print("="*50)
print("RADIOGRAFÍA DEL DATASET: DB_AFAC")
print("="*50)

info_afac = pd.DataFrame({
    'Tipo de Dato': df_afac.dtypes,
    'Valores Nulos': df_afac.isnull().sum(),
    '% Nulo': (df_afac.isnull().sum() / len(df_afac) * 100).round(2),
    'Valores Únicos': df_afac.nunique()
})

display(info_afac)

print("\nLista exacta de columnas:")
print(df_afac.columns.tolist())

RADIOGRAFÍA DEL DATASET: DB_AFAC


,Tipo de Dato,Valores Nulos,% Nulo,Valores Únicos
Año,int64,0,0.0,10
Tipo,object,0,0.0,2
Servicio,object,0,0.0,2
Reg,object,0,0.0,7
Aerolinea,object,0,0.0,195
Id_mes,int64,0,0.0,12
Mes,object,0,0.0,12
Pasajeros,int64,0,0.0,8537
fecha,int64,0,0.0,120
Tipo.1,object,0,0.0,2



Lista exacta de columnas:
['Año', 'Tipo', 'Servicio', 'Reg', 'Aerolinea', 'Id_mes', 'Mes', 'Pasajeros', 'fecha', 'Tipo.1', 'Servicio.1', 'Aerolínea']


## 3. Limpieza de datos

### 3.1 Eliminación de columnas redundantes y renombramiento

Se eliminan las columnas que no aportan información adicional al modelo:
`fecha` (redundante con `Año` y `MesNum`), `Tipo.1` y `Servicio.1`
(duplicados de `Tipo` y `Servicio`), `Aerolínea` (duplicado de `Aerolinea`)
y `Mes` (redundante con `Id_mes` numérico).

Posteriormente se renombran las columnas restantes para unificar la
nomenclatura con los demás datasets del proyecto: `Id_mes → MesNum`,
`Reg → Region_Aerolinea`, `Tipo → Tipo_Vuelo` y `Pasajeros → Valor`.

In [ ]:
print("Iniciando limpieza estructural...")

cols_a_eliminar = ['fecha', 'Tipo.1', 'Servicio.1', 'Aerolínea', 'Mes']
df_afac.drop(columns=[col for col in cols_a_eliminar if col in df_afac.columns], inplace=True)

nuevos_nombres = {
    'Id_mes': 'MesNum',
    'Reg': 'Region_Aerolinea',
    'Tipo': 'Tipo_Vuelo',
    'Pasajeros': 'Valor'
}
df_afac.rename(columns=nuevos_nombres, inplace=True)

print(f"Columnas resultantes: {df_afac.columns.tolist()}")

Iniciando limpieza estructural...
Columnas resultantes: ['Año', 'Tipo_Vuelo', 'Servicio', 'Region_Aerolinea', 'Aerolinea', 'MesNum', 'Valor']


### 3.2 Filtrado de registros y segmentación del dataset

Se aplican tres operaciones de filtrado:

1. **Eliminación de vuelos sin pasajeros**: se descartan registros con
   `Valor = 0`, que corresponden a vuelos de carga pura o cancelados
   y no son relevantes para el análisis de flujo turístico.
2. **Separación de vuelos nacionales e internacionales**: dado que el
   proyecto se enfoca en turismo receptor internacional, los vuelos
   nacionales se separan en un dataframe independiente (`df_nacional`)
   y se trabaja únicamente con los internacionales (`df_afac_int`).
3. **Generación del dataset modelo**: se excluyen los años de pandemia
   (2020–2021) y el año en curso incompleto (2026), conservando el
   periodo de comportamiento normal para el entrenamiento del modelo.

In [ ]:
filas_antes = len(df_afac)
df_afac = df_afac[df_afac['Valor'] > 0].copy()
print(f"Vuelos con 0 pasajeros eliminados: {filas_antes - len(df_afac)}")

df_nacional = df_afac[df_afac['Tipo_Vuelo'] == 'NAC'].copy()
df_afac_int = df_afac[df_afac['Tipo_Vuelo'] != 'NAC'].copy()
print(f"Vuelos Nacionales separados: {len(df_nacional):,}")
print(f"Vuelos Internacionales retenidos: {len(df_afac_int):,}")

años_a_excluir = [2020, 2021, 2026]
df_afac_modelo = df_afac_int[~df_afac_int['Año'].isin(años_a_excluir)].copy()
print(f"Filas listas para el modelo internacional: {len(df_afac_modelo):,}")

Vuelos con 0 pasajeros eliminados: 8917
Vuelos Nacionales separados: 1,645
Vuelos Internacionales retenidos: 8,242
Filas listas para el modelo internacional: 6,939


## 4. Normalización de variables categóricas

Las columnas de texto (`Tipo_Vuelo`, `Servicio`, `Region_Aerolinea`, `Aerolinea`)
se homogenizan aplicando conversión a minúsculas, eliminación de acentos
mediante normalización Unicode (NFKD) y supresión de espacios en blanco.
Al finalizar se presenta el estado definitivo del dataset: dimensiones,
estructura de columnas y muestra de datos normalizados.

In [ ]:
print("Iniciando normalización de texto...")

columnas_texto = ['Tipo_Vuelo', 'Servicio', 'Region_Aerolinea', 'Aerolinea']

for col in columnas_texto:

    df_afac_modelo[col] = df_afac_modelo[col].str.lower()

    df_afac_modelo[col] = df_afac_modelo[col].str.normalize('NFKD')\
                                             .str.encode('ascii', errors='ignore')\
                                             .str.decode('utf-8')

    df_afac_modelo[col] = df_afac_modelo[col].str.strip()

print("="*50)
print("ESTADO FINAL DEL DATASET AEROLÍNEAS (MODELO INT)")
print("="*50)
print(f"Forma final: {df_afac_modelo.shape[0]:,} filas x {df_afac_modelo.shape[1]} columnas")
display(df_afac_modelo.head())

Iniciando normalización de texto...
ESTADO FINAL DEL DATASET AEROLÍNEAS (MODELO INT)
Forma final: 6,939 filas x 7 columnas


,Año,Tipo_Vuelo,Servicio,Region_Aerolinea,Aerolinea,MesNum,Valor
0,2016,int,flet,can,air transat,1,52930
2,2016,int,flet,can,sunwing airlines,1,33932
4,2016,int,flet,csa,avianca,1,106
6,2016,int,flet,csa,copa airlines,1,318
7,2016,int,flet,csa,lacsa (lineas aereas costarricences),1,523


## 5. Exportación del dataset limpio

El dataset modelo de aerolíneas, filtrado a vuelos internacionales,
sin años atípicos y con variables normalizadas, se exporta en formato CSV
para su integración con los demás datasets del proyecto.

In [ ]:
print("Guardando BD_AFAC_modelo.csv...")
df_afac_modelo.to_csv('BD_AFAC_modelo.csv', index=False)

from google.colab import files
files.download('BD_AFAC_modelo.csv')



Guardando BD_AFAC_modelo.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>